# Prediction territories (SegFormer habitats), minimal version

- **Composites:** empty camera-trap frames (id_test and test, sampled evenly over the six habitats below) with an
  animal cut-out pasted on one cell of a 3x3 grid (3 cut-outs per species group); the ResNet-50 classifies every
  composite (`src/run_composites.py`).
- **Habitat of a camera:** SegFormer-B2 (`src/measure_background.py`): share of the outdoor background without
  trees / shrubs; **open** above 2/3, **forest** below 1/3, **mixed** in between. Day / night (infrared) comes from
  the frame.
- **A dot** = one background frame, placed at the openness of its camera (x) and its brightness (y), coloured by
  the most common outcome over the three cut-outs (ties: the better outcome): correct (exact species), right group but other species, other group, or "empty".

If the model recognised the animal, the whole map would be blue. If it reads the place, whole habitats take the same
wrong colour: "territories".

In [ ]:
import base64
import io

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from PIL import Image
from plotly.subplots import make_subplots

from common import load_config
from common.data import IWildCamImages
from src.composite import CELL_NAMES, FOREST_T, OPEN_T, PAIR_B, load_cutouts, paste

cfg = load_config()

# the backgrounds: openness and habitat of their camera (SegFormer, src/composite.py: camera_habitats),
# light = day / night (infrared); sampled evenly over these six habitats by src/run_composites.py
bg = pd.read_parquet(PAIR_B / "results" / "backgrounds.parquet")
bg["habitat"] = bg["light"] + " · " + bg["habitat"]
HABITATS = [f"{light} · {veg}" for light in ["day", "night"] for veg in ["open", "mixed", "forest"]]

res = pd.read_parquet(PAIR_B / "results" / "composites.parquet").rename(columns={"y": "species_y"})
res = res.merge(bg[["bg_id", "light", "location", "habitat"]], on="bg_id")
res["correct"] = res["outcome"] == "exact species"
res["empty"] = res["outcome"] == "empty"
ANIMALS = list(pd.read_csv(PAIR_B / "cutouts" / "cutouts.csv")["group"].unique())
print(f"{len(bg)} backgrounds, {len(ANIMALS)} animals, {len(res):,} composites")

In [ ]:
INK = "#1f1f1e"
INK_2 = "#52514e"
MUTED = "#8a8984"
GRID = "#e7e6e1"
SURFACE = "#ffffff"
FONT = "Inter, DejaVu Sans, Arial, sans-serif"  # plotly picks the first one it finds
OUTCOMES = ["exact species", "same group", "other group", "empty"]
LABEL = {
    "exact species": "correct (exact species)",
    "same group": "right group, other species",
    "other group": "other group",
    "empty": "predicted empty",
}
COLOR = {
    "exact species": "#2a78d6",
    "same group": "#1baf7a",
    "other group": "#eb6834",
    "empty": "#c9c8c2",
}
VEGETATION = ["open", "mixed", "forest"]
HABITAT_COLOR = {"open": "#eda100", "mixed": "#8c9440", "forest": "#008300"}  # checked with the validator too

# the 3x3 grid of paste positions (row, col), drawn as arrows
ARROWS = {
    (0, 0): "↖", (0, 1): "↑", (0, 2): "↗",
    (1, 0): "←", (1, 1): "●", (1, 2): "→",
    (2, 0): "↙", (2, 1): "↓", (2, 2): "↘",
}

axis_style = dict(gridcolor=GRID, linecolor=GRID, zeroline=False, ticks="", tickfont=dict(color=INK_2))
pio.templates["course"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=15, color=INK),
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        title=dict(x=0.01, xanchor="left", font=dict(size=20, color=INK)),
        xaxis=axis_style,
        yaxis=axis_style,
        legend=dict(font=dict(color=INK_2), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor=SURFACE, bordercolor=GRID, font=dict(family=FONT, color=INK, size=15)),
        margin=dict(l=70, r=30, t=90, b=60)
    )
)
pio.templates.default = "simple_white+course"


def subtitle(text):
    """Smaller grey text behind a plotly title."""
    return f"<span style='font-size:16px;color:{INK_2}'>{text}</span>"


def legend_dot(name, **marker):
    """An empty trace that only adds a dot to the legend."""
    return go.Scatter(x=[None], y=[None], mode="markers", name=name, marker=dict(size=11, **marker))


In [ ]:
cutouts = load_cutouts()
example_file = "909a9066-21bc-11ea-a13a-137349068a90.jpg"  # picked by hand: clear daytime savanna frame
example_image, _ = IWildCamImages(pd.DataFrame({"filename": [example_file]}), cfg)[0]
example_img = example_image[None]  # as a batch of one


def select(animal, row=None, col=None, variant="all"):
    """The composites of one animal, optionally only one paste position and / or one cut-out."""
    d = res[res["group"] == animal]
    if row is not None:
        d = d[(d["row"] == row) & (d["col"] == col)]
    if variant != "all":
        d = d[d["variant"] == variant]
    return d


def per_background(d):
    """One row per background: the most common outcome over the cut-outs and the answers (for the hover)."""
    counts = pd.crosstab(d["bg_id"], d["outcome"]).reindex(columns=OUTCOMES, fill_value=0)
    answers = d.sort_values("variant").groupby("bg_id")["pred_species"].agg(" · ".join)
    dots = pd.DataFrame({
        "outcome": counts.idxmax(axis=1),  # ties: the first = better outcome
        "answers": answers,
    })
    return dots.join(bg.set_index("bg_id")[["location", "habitat"]])


def map_traces(dots, size=8, visible=True):
    """One scatter, each dot coloured by its outcome."""
    return [go.Scatter(
        x=dots["x"].round(3),  # 3 decimals: smaller figure
        y=dots["y"].round(3),
        mode="markers",
        showlegend=False,
        visible=visible,
        marker=dict(color=[COLOR[o] for o in dots["outcome"]], size=size, line=dict(color=SURFACE, width=0.8)),
        customdata=np.stack([dots["habitat"], dots["location"].astype(str), dots["answers"]], axis=-1),
        hovertemplate="%{customdata[0]}<br>camera %{customdata[1]}<br>%{customdata[2]}<extra></extra>"
    )]


def legend_traces():
    """The four outcomes, for the legend only."""
    return [legend_dot(LABEL[o], color=COLOR[o]) for o in OUTCOMES]

## Prediction-territory explorer

**Controls:** the animal in the dropdown, the paste position with the arrow keypad (the nine buttons are the 3x3
grid over the frame, ● = centre). The map: x = SegFormer openness of the camera (shaded bands = habitats), y =
brightness of the frame, one row for day and one for night. Right: the animal on an example frame and the
outcome per habitat (share correct and the most common wrong answer). Hover a dot for its habitat, camera and the
three answers.

In [ ]:
def jpeg_uri(img):
    """Image array -> inline jpeg (much smaller in the figure than a png)."""
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format="JPEG", quality=70)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def explorer(animals, start=("elephant", (2, 1))):
    """Animal dropdown + 3x3 position keypad, all inside one plotly figure (no ipywidgets).

    Plotly buttons cannot combine several choices, so each control changes something else: the dropdown
    switches which traces are visible (every animal has its own traces), the keypad moves the x range. The nine
    positions sit next to each other in data space (map, example image and habitat bars).
    """
    positions = list(ARROWS)

    # ---- layout in data coordinates ----

    # one window of the map (data units); the windows of the 9 positions sit next to each other, map_w apart
    x_min, x_max = 0.0, 5.07
    map_w = x_max - x_min + 1.0
    img_w = 500  # same for the example images
    block = len(HABITATS) + 2  # and the habitat bars: one block of rows per position

    # y range with the same scale as x: plot area in px = figure size minus margins, map = 63% of the width
    plot_w = 1450 - 70 - 190
    plot_h = 800 - 190 - 70
    y_top = 0.0
    y_half = (x_max - x_min) * plot_h / (plot_w * (1 - 0.17) * 0.63) / 2
    y_mid = (y_top - 2.81 + 1.0) / 2
    view_y = [y_mid - y_half, y_mid + y_half]

    # x = openness of the camera, y = brightness of the frame, one row for day and one for night
    lo = min(bg["openness"].min(), 0) - 0.03
    hi = max(bg["openness"].max(), 1) + 0.03
    ox0 = x_min + 0.75  # openness axis from ox0 to ox1
    ox1 = x_max - 0.1
    b_bottom = view_y[0] + 0.55
    b_top = y_top
    row_h = (b_top - b_bottom - 0.3) / 2
    row_y = {"day": b_top - row_h, "night": b_bottom}  # lower edge of each row

    def to_ox(o):
        """Openness -> x in the openness view."""
        return ox0 + (o - lo) / (hi - lo) * (ox1 - ox0)

    b = bg.set_index("bg_id")
    jitter = np.random.default_rng(0).uniform(-0.007, 0.007, len(b))  # frames of a camera share x
    open_x = to_ox(b["openness"] + jitter)
    open_y = b["light"].map(row_y) + b["brightness"].clip(0, 1) * row_h

    fig = make_subplots(
        rows=2,
        cols=2,
        column_widths=[0.63, 0.37],
        row_heights=[0.36, 0.64],
        horizontal_spacing=0.17,
        vertical_spacing=0.1,
        specs=[[{"rowspan": 2}, {}], [None, {}]],  # left: map, right: example frame above the habitat bars
        subplot_titles=["", "on an example frame (cut-out 1 of 3)", "outcome per habitat (name = most common wrong answer)"]
    )
    fig.update_annotations(font=dict(size=15, color=INK_2))
    owner = []  # animal a trace belongs to (None = always visible)

    def add(trace, row, col, animal=None):
        fig.add_trace(trace, row=row, col=col)
        owner.append(animal)

    # ---- fixed labels of the map, once per position window ----

    xs, ys, texts = [], [], []

    def label(x, y, text):
        xs.append(x)
        ys.append(y)
        texts.append(text)

    bands = [("forest", lo, FOREST_T), ("mixed", FOREST_T, OPEN_T), ("open", OPEN_T, hi)]

    for p in range(9):
        dx = p * map_w  # x offset of the window

        # habitat bands, labels and the axis (as text, repeated in every window)
        for veg, o0, o1 in bands:
            label((to_ox(o0) + to_ox(o1)) / 2 + dx, b_top + 0.13, veg)
            for light in ["day", "night"]:
                fig.add_shape(
                    type="rect",
                    x0=to_ox(o0) + dx,
                    x1=to_ox(o1) + dx,
                    y0=row_y[light],
                    y1=row_y[light] + row_h,
                    fillcolor=HABITAT_COLOR[veg],
                    opacity=0.08,
                    line_width=0,
                    layer="below",
                    row=1,
                    col=1
                )
        for light in ["day", "night"]:
            label(x_min + 0.3 + dx, row_y[light] + row_h / 2, f"<b>{light.upper()}</b>")
            label(ox0 - 0.12 + dx, row_y[light] + row_h - 0.08, "bright")
            label(ox0 - 0.12 + dx, row_y[light] + 0.08, "dark")
        for t in [0, 0.25, 0.5, 0.75, 1]:
            label(to_ox(t) + dx, b_bottom - 0.14, f"{t:.0%}")
        label((ox0 + ox1) / 2 + dx, b_bottom - 0.42, "openness of the camera = share of background without trees →")

    add(
        go.Scatter(
            x=xs,
            y=ys,
            text=texts,
            mode="text",
            textfont=dict(size=14, color=INK_2),
            hoverinfo="skip",
            showlegend=False
        ),
        row=1,
        col=1
    )

    # ---- the traces of each animal, for all 9 positions ----

    for animal in animals:
        vis = animal == start[0]
        status = []  # the line above the map
        bars = {o: [] for o in OUTCOMES}  # outcome -> (share, y, habitat) of its bars
        bar_labels = []  # (y, text) right of the bars

        for p, (r, c) in enumerate(positions):
            dx = p * map_w
            d = select(animal, r, c)
            dots = per_background(d)

            # the map
            dots = dots.assign(x=open_x.reindex(dots.index) + dx, y=open_y.reindex(dots.index))
            for t in map_traces(dots, size=7, visible=vis):
                add(t, 1, 1, animal)

            same_group = (d["outcome"] == "same group").mean()
            status.append(
                f"<b>{animal}</b>, pasted {CELL_NAMES[(r, c)]}: {d['correct'].mean():.0%} correct · "
                f"{same_group:.0%} right group, other species · {d['empty'].mean():.0%} empty"
            )

            # the first cut-out on the example frame
            comp = paste(example_img, cutouts[animal, 0], r, c)[0].permute(1, 2, 0).numpy()
            comp = np.asarray(Image.fromarray(comp).resize((448, 252)))
            image = go.Image(source=jpeg_uri(comp), x0=p * img_w, y0=0, visible=vis, hoverinfo="skip")
            add(image, 1, 2, animal)

            # habitat bars: share of each outcome, and the most common wrong answer
            share = pd.crosstab(d["habitat"], d["outcome"], normalize="index")
            share = share.reindex(index=HABITATS, columns=OUTCOMES, fill_value=0)
            wrong = d[~d["correct"] & ~d["empty"]]
            most_wrong = wrong.groupby("habitat")["pred_species"].agg(lambda s: s.value_counts().index[0])
            for k, h in enumerate(HABITATS):
                y = p * block + k
                for o in OUTCOMES:
                    bars[o].append((share.loc[h, o], y, h))
                text = f"{share.loc[h, 'exact species']:.0%} ✓"
                if h in most_wrong:
                    text += f" · {most_wrong[h]}"
                bar_labels.append((y, text))

        status_line = go.Scatter(  # the line above the map
            x=[x_min + p * map_w for p in range(9)],
            y=[y_top + 0.75] * 9,
            text=status,
            mode="text",
            textposition="middle right",
            textfont=dict(size=16, color=INK),
            cliponaxis=False,  # longer than the map at this font size
            hoverinfo="skip",
            showlegend=False,
            visible=vis
        )
        add(status_line, 1, 1, animal)

        for o in OUTCOMES:
            shares, bar_y, habitats = zip(*bars[o])
            bar = go.Bar(
                x=shares,
                y=bar_y,
                customdata=habitats,
                orientation="h",
                showlegend=False,
                visible=vis,
                name=LABEL[o],
                marker=dict(color=COLOR[o], line=dict(color=SURFACE, width=2)),
                hovertemplate="%{customdata}: %{x:.0%} " + LABEL[o] + "<extra></extra>"
            )
            add(bar, 2, 2, animal)

        label_y, label_text = zip(*bar_labels)
        bar_text = go.Scatter(
            x=[1.03] * len(label_y),
            y=label_y,
            text=label_text,
            mode="text",
            textposition="middle right",
            textfont=dict(size=13, color=INK_2),
            cliponaxis=False,  # the labels may run into the right margin
            hoverinfo="skip",
            showlegend=False,
            visible=vis
        )
        add(bar_text, 2, 2, animal)

    for t in legend_traces():
        add(t, 1, 1)

    # ---- axes ----

    # map: no scaleanchor here, the y range is set to the right scale above
    fig.update_xaxes(visible=False, row=1, col=1)
    fig.update_yaxes(visible=False, range=view_y, row=1, col=1)

    # example frame: y in pixels, from the top
    fig.update_xaxes(visible=False, row=1, col=2)
    fig.update_yaxes(visible=False, range=[252, 0], row=1, col=2)

    # habitat bars
    fig.update_xaxes(
        range=[0, 1.75],
        tickvals=[0, 0.25, 0.5, 0.75, 1],
        tickformat=".0%",
        showgrid=False,
        row=2,
        col=2
    )
    fig.update_yaxes(
        tickvals=[p * block + k for p in range(9) for k in range(len(HABITATS))],
        ticktext=HABITATS * 9,
        showgrid=False,
        row=2,
        col=2
    )

    # ---- controls ----

    # the axes the buttons move (plotly names like "xaxis", "yaxis2")
    map_x = fig.get_subplot(1, 1).xaxis.plotly_name
    img_x = fig.get_subplot(1, 2).xaxis.plotly_name
    bars_y = fig.get_subplot(2, 2).yaxis.plotly_name

    def window(p):
        """Axis ranges that show position p."""
        return {
            f"{map_x}.range": [x_min + p * map_w, x_max + p * map_w],
            f"{img_x}.range": [p * img_w, p * img_w + 448],
            f"{bars_y}.range": [p * block + len(HABITATS) - 0.5, p * block - 0.5],
        }

    menu_style = dict(bgcolor=SURFACE, bordercolor=GRID, font=dict(size=15, color=INK))
    no_pad = dict(t=0, b=0, l=0, r=0)

    # one menu per key, placed on an exact 3x3 grid; monospace font so all keys get the same width
    key_w, key_h = 48 + 3, 33 + 3  # key size in px (measured in the browser) + gap
    keypad = []
    for r in range(3):
        for c in range(3):
            key = dict(label=f" {ARROWS[(r, c)]} ", method="relayout", args=[window(r * 3 + c)])
            keypad.append(dict(
                type="buttons",
                showactive=False,
                x=0.2 + c * key_w / plot_w,
                y=1.225 - r * key_h / plot_h,
                xanchor="left",
                yanchor="top",
                pad=no_pad,
                bgcolor=SURFACE,
                bordercolor=GRID,
                font=dict(size=16, color=INK, family="DejaVu Sans Mono, monospace"),
                buttons=[key]
            ))

    animal_buttons = []
    for a in animals:
        visible = [o is None or o == a for o in owner]
        animal_buttons.append(dict(label=a, method="restyle", args=[{"visible": visible}]))
    dropdown = dict(
        type="dropdown",
        active=animals.index(start[0]),
        x=0,
        y=1.225,
        xanchor="left",
        yanchor="top",
        buttons=animal_buttons,
        **menu_style
    )

    def control_label(text, x):
        return dict(
            text=text,
            x=x,
            y=1.27,
            xref="paper",
            yref="paper",
            showarrow=False,
            xanchor="left",
            font=dict(size=14, color=MUTED)
        )

    fig.update_layout(
        updatemenus=[dropdown] + keypad,
        barmode="stack",
        bargap=0.3,
        width=1450,
        height=800,
        margin=dict(t=190, r=190, b=70),
        legend=dict(orientation="h", x=0, y=-0.06),
        title=dict(text="Prediction-territory explorer", y=0.985),
        annotations=list(fig.layout.annotations) + [
            control_label("animal", 0),
            control_label("paste position in the frame", 0.2),
        ]
    )
    # bigger example frame: it also takes the empty gap between the map and the right column
    # (px of the plot area -> paper; height from the frame's 448 x 252 aspect), the bars move down
    frame_x = [(790 - 70) / plot_w, (1250 - 70) / plot_w]
    frame_y = [1 - (1250 - 790) * 252 / 448 / plot_h, 1]
    bars_top = frame_y[0] - 0.1
    fig.update_xaxes(domain=frame_x, row=1, col=2)
    fig.update_yaxes(domain=frame_y, row=1, col=2)
    fig.update_yaxes(domain=[0, bars_top], row=2, col=2)
    fig.for_each_annotation(lambda a: a.update(x=sum(frame_x) / 2, y=1) if a.text.startswith("on an example") else None)
    fig.for_each_annotation(lambda a: a.update(y=bars_top) if a.text.startswith("outcome per habitat") else None)
    fig.update_layout(window(positions.index(start[1])))  # show the start position
    return fig


explorer(ANIMALS).show()

In [ ]:
# a few interesting selections of the explorer as static figures (controls removed), saved as pdf
SNAPSHOTS = [  # (animal, paste position, why it is interesting)
    ("elephant", (2, 1), "Savanna animal: right on open land, a great argus (forest bird) in forest"),
    ("cattle/buffalo", (2, 2), "Biggest open − forest gap by day (+83 points)"),
    ("porcupine", (1, 1), "Forest animal: at night right in forest, almost never on open land"),
    ("big cat", (1, 1), "Forest animal by day: recognised in forest and mixed, not on open land"),
    ("bird", (0, 0), "Top left: 91% predicted empty, the bird is not seen at all"),
]
SNAP_DIR = PAIR_B / "figures" / "territory_snapshots"
SNAP_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_LABELS = {"animal", "paste position in the frame"}

for animal, cell, why in SNAPSHOTS:
    fig = explorer([animal], start=(animal, cell))
    fig.layout.updatemenus = []  # update_layout(updatemenus=[]) would keep the old menus
    fig.update_layout(
        annotations=[a for a in fig.layout.annotations if a.text not in CONTROL_LABELS],
        title=dict(text=why, y=0.97),
        margin=dict(t=110)
    )
    name = f"{animal}_{CELL_NAMES[cell]}".replace("/", "_").replace(" ", "_")
    fig.write_image(SNAP_DIR / f"{name}.pdf", width=fig.layout.width, height=fig.layout.height)
    fig.show(renderer="png", width=fig.layout.width, height=fig.layout.height)
print("saved to", SNAP_DIR.relative_to(PAIR_B), ":", sorted(f.name for f in SNAP_DIR.glob("*.pdf")))

## Observations

Shares correct over all paste positions and cut-outs.

- **Savanna animals are recognised on open land, much less in forest.** By day: elephant 93% open vs. 45% forest,
  cattle 80% vs. 19%, zebra 100% vs. 49%. The right (open) end of the axis turns blue, the left (forest) end orange.
- **Forest animals the other way round.** Porcupine at night 84% in forest vs. 1% on open land, big cat by day
  38% vs. 2%.
- **The wrong answers fit the place.** By day in forest the most common wrong answers are *great argus* and
  *sun bear*, on open land *vulturine guineafowl* and *giraffe*. The elephant pasted at the bottom becomes a great argus
  in forest and mixed, a vulturine guineafowl on open land, by day and at night.
- **Top row = mostly grey.** Pasted in the top row, 49% of the composites are predicted empty (middle row 21%,
  bottom row 29%).
- **No territories without recognition.** Deer, agouti, dog/fox, bird and antelope are correct in at most 6% of the
  composites in every habitat; their maps are orange and grey everywhere.
- **Caveat:** open land by day has only 47 frames from 7 cameras (all other habitats 77–80 frames), there are few
  open cameras with empty frames in id_test / test.
